# Feature Engineering: TF-IDF & Word2Vec Pipeline
## Research: Requirement Prioritization (Phase 1)
**Local Execution & Leak-Free Learning-to-Rank (LTR)**

Pipeline ini mengimplementasikan ekstraksi fitur teks, seleksi fitur, penanganan kebocoran data (*data leakage prevention*), dan pembagian dataset yang ketat:
1. **Pemuatan Data Lokal & Natural Sorting**: Membaca dataset bersih TF-IDF dan Word2Vec, mengurutkan nomor proyek ($P1 \dots P30$) dan requirement ($REQ-01 \dots$) secara stabil.
2. **Standardisasi Target LTR (`relevance_grade`)**: Menormalisasi target `priority_score` per proyek menjadi 5 tingkat relevansi diskrit ($0 \dots 4$) untuk algoritma LTR (NDCG).
3. **Jalur 1 — TF-IDF + Chi-Square Selection**: `sublinear_tf=True`, n-gram (1,2), dan `SelectKBest(chi2, k=100)` terhadap `relevance_grade` ($df \ge 2$).
4. **Jalur 2 — Word2Vec**: Arsitektur Skip-gram (`sg=1`), 100 dimensi, 50 epoch, hash deterministik, serta L2-normalization pada rata-rata vektor kalimat.
5. **Pemisahan Dataset Anti-Leakage**:
   - **PureText Dataset (104 kolom)**: Khusus perbandingan representasi teks murni (TF-IDF vs Word2Vec vs SBERT) tanpa membocorkan formula prioritas.
   - **Fusion Dataset (108 kolom)**: Menyertakan fitur numerik domain dalam skala aslinya (*unscaled*) untuk eksperimen fusi fitur & studi ablasi.
   - **EDA / Ready Dataset (113 kolom)**: Menyertakan versi Z-score (*StandardScaler*) untuk visualisasi tim.
6. **Ekspor Lokal**: Menyimpan semua file ke direktori `./output_datasets/`.

In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.model_selection import GroupKFold
from gensim.models import Word2Vec

# ── 1. Reproducibility Seeds ────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ── 2. Local Path Configuration ─────────────────────────────────────────────
# Resolusi otomatis folder dataset lokal
CANDIDATE_DIRS = [Path("./dataset"), Path("./datasets"), Path(".")]
INPUT_DIR = next((d for d in CANDIDATE_DIRS if (d / "Dataset_Cleaned_TFIDF.csv").exists()), Path("./dataset"))

OUTPUT_DIR = Path("./output_datasets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Environment & Local Paths Ready:")
print(f"  * Input Directory  : {INPUT_DIR.resolve()}")
print(f"  * Output Directory : {OUTPUT_DIR.resolve()}")
print(f"  * Random Seed      : {SEED}")

Environment & Local Paths Ready:
  * Input Directory  : C:\Users\EdselSpth\Downloads\Riset TEMP\dataset
  * Output Directory : C:\Users\EdselSpth\Downloads\Riset TEMP\output_datasets
  * Random Seed      : 42


---
## 1. Load Data Lokal, Natural Sorting, & Pembuatan `relevance_grade`

Langkah ini membaca dataset hasil preprocessing, menerapkan **Natural Sorting** agar urutan proyek numerik ($P1, P2 \dots P30$) dan ID requirement ($REQ-01, REQ-02 \dots$) stabil 100% serta sinkron dengan dataset embedding eksternal (SBERT), lalu menghitung target relevansi terstandarisasi per proyek (**`relevance_grade`** $0 \dots 4$).

In [2]:
# ── 1. Load Input Datasets ──────────────────────────────────────────────────
tfidf_file = INPUT_DIR / "Dataset_Cleaned_TFIDF.csv"
w2v_file   = INPUT_DIR / "Dataset_Cleaned_W2V.csv"

if not tfidf_file.exists() or not w2v_file.exists():
    raise FileNotFoundError(
        f"File dataset tidak ditemukan di '{INPUT_DIR}'. "
        f"Pastikan 'Dataset_Cleaned_TFIDF.csv' dan 'Dataset_Cleaned_W2V.csv' tersedia."
    )

df_tfidf = pd.read_csv(tfidf_file)
df_w2v   = pd.read_csv(w2v_file)

# ── 2. Natural Sorting Helper ───────────────────────────────────────────────
def natural_sort(df: pd.DataFrame) -> pd.DataFrame:
    """Urutkan project_id (P1..P30) dan id (REQ-01..REQ-952) secara numerik stabil."""
    proj_num = df['project_id'].astype(str).str.extract(r'(\d+)')[0].astype(int)
    req_num  = df['id'].astype(str).str.extract(r'(\d+)')[0].astype(int)
    return (
        df.assign(_p=proj_num, _r=req_num)
        .sort_values(by=['_p', '_r'], kind='stable')
        .drop(columns=['_p', '_r'])
        .reset_index(drop=True)
    )

df_tfidf = natural_sort(df_tfidf)
df_w2v   = natural_sort(df_w2v)

# ── 3. Cross-Dataset Validation Gates ───────────────────────────────────────
assert len(df_tfidf) == 952, f"Expected 952 rows for TF-IDF, got {len(df_tfidf)}"
assert len(df_w2v) == 952, f"Expected 952 rows for Word2Vec, got {len(df_w2v)}"
assert df_tfidf.isnull().sum().sum() == 0, "Missing values found in TF-IDF dataset!"
assert df_w2v.isnull().sum().sum() == 0, "Missing values found in Word2Vec dataset!"
assert (df_tfidf['id'] == df_w2v['id']).all(), "Mismatch in 'id' between TF-IDF and Word2Vec!"
assert (df_tfidf['project_id'] == df_w2v['project_id']).all(), "Mismatch in 'project_id'!"
assert (df_tfidf['rank'] == df_w2v['rank']).all(), "Mismatch in 'rank'!"
assert (df_tfidf['priority_score'] == df_w2v['priority_score']).all(), "Mismatch in 'priority_score'!"

print("[Gate 1 Passed] Integritas dan keselarasan data 100% valid.")

# ── 4. Categorical Normalization (FR = 1, NFR = 0) ───────────────────────────
df_tfidf['type_encoded'] = (df_tfidf['type'].astype(str).str.strip().str.upper() == 'FR').astype(int)
df_w2v['type_encoded']   = df_tfidf['type_encoded']

# ── 5. Standardized Relevance Grade Calculation for LTR ─────────────────────
def compute_within_project_relevance(group_scores: pd.Series, n_bins: int = 5) -> pd.Series:
    """
    Petakan priority_score per proyek ke dalam integer 0..4 (5 tingkat relevansi).
    Skor tertinggi = grade 4 (Top 20%), terendah = grade 0.
    Cocok untuk LambdaRank (LightGBM/XGBoost/CatBoost) dan metrik NDCG.
    """
    pct = group_scores.rank(ascending=True, method="average", pct=True)
    return np.floor(pct * n_bins).clip(0, n_bins - 1).astype(int)

df_tfidf['relevance_grade'] = (
    df_tfidf.groupby('project_id')['priority_score']
    .transform(compute_within_project_relevance)
)
df_w2v['relevance_grade'] = df_tfidf['relevance_grade']

print(f"Total baris        : {len(df_tfidf)} baris")
print(f"Total proyek unik  : {df_tfidf['project_id'].nunique()} proyek (P1..P30)")
print("Distribusi relevance_grade:")
print(df_tfidf['relevance_grade'].value_counts().sort_index().to_dict())

[Gate 1 Passed] Integritas dan keselarasan data 100% valid.
Total baris        : 952 baris
Total proyek unik  : 30 proyek (P1..P30)
Distribusi relevance_grade:
{0: 153, 1: 217, 2: 180, 3: 198, 4: 204}


---
## 2. Feature Engineering Jalur 1 — TF-IDF & Chi-Square Selection

Ekstraksi representasi statistik teks menggunakan **TF-IDF dengan peredaman sublinear** (`sublinear_tf=True`), n-gram unigram + bigram (`ngram_range=(1, 2)`), dan batasan frekuensi minimal dokumen (`min_df=2`). Selanjutnya, fitur disaring menjadi **100 fitur terbaik** menggunakan **Chi-Square (`SelectKBest`)** yang dikorelasikan terhadap `relevance_grade` (bukan `rank` mentah) untuk mencegah pemilihan kata *spurious* dengan frekuensi dokumen tunggal ($df=1$).

In [3]:
# ── 1. TF-IDF Vectorization ─────────────────────────────────────────────────
text_tfidf = df_tfidf['cleaned_text'].fillna('')

tfidf_vectorizer = TfidfVectorizer(
    min_df=2,
    ngram_range=(1, 2),
    sublinear_tf=True
)
tfidf_matrix = tfidf_vectorizer.fit_transform(text_tfidf)
print(f"Matriks TF-IDF mentah : {tfidf_matrix.shape[0]} dokumen x {tfidf_matrix.shape[1]} vocab")

# ── 2. Chi-Square Feature Selection (k=100) on relevance_grade ──────────────
k_features = min(100, tfidf_matrix.shape[1])
chi2_selector = SelectKBest(score_func=chi2, k=k_features)
tfidf_selected = chi2_selector.fit_transform(tfidf_matrix, df_tfidf['relevance_grade'])

selected_indices = chi2_selector.get_support(indices=True)
all_feature_names = tfidf_vectorizer.get_feature_names_out()

# Format nama kolom: tfidf_<term> (spasi diganti underscore)
selected_feature_names = [f"tfidf_{all_feature_names[i].replace(' ', '_')}" for i in selected_indices]
df_tfidf_features = pd.DataFrame(tfidf_selected.toarray(), columns=selected_feature_names)

# ── 3. Health Check: Document Frequency of Selected Features ────────────────
doc_freqs = (tfidf_matrix[:, selected_indices] > 0).sum(axis=0).A1
min_df = int(doc_freqs.min())
med_df = float(np.median(doc_freqs))
mean_df = float(doc_freqs.mean())

print(f"[OK] TF-IDF Feature Selection selesai: {df_tfidf_features.shape[1]} fitur terpilih.")
print(f"   * Min Document Frequency    : {min_df} (Bebas dari fitur df=1)")
print(f"   * Median Document Frequency : {med_df:.1f}")
print(f"   * Mean Document Frequency   : {mean_df:.2f}")

assert min_df >= 2, f"Peringatan: Ditemukan fitur dengan document frequency < 2 (min_df={min_df})"
print(f"Contoh 8 fitur TF-IDF terpilih: {selected_feature_names[:8]}")

Matriks TF-IDF mentah : 952 dokumen x 1865 vocab
[OK] TF-IDF Feature Selection selesai: 100 fitur terpilih.
   * Min Document Frequency    : 2 (Bebas dari fitur df=1)
   * Median Document Frequency : 4.0
   * Mean Document Frequency   : 7.17
Contoh 8 fitur TF-IDF terpilih: ['tfidf_access_authorized', 'tfidf_admin_finance', 'tfidf_alert', 'tfidf_allergic', 'tfidf_applicable', 'tfidf_assessment', 'tfidf_associate_order', 'tfidf_authorization']


---
## 3. Feature Engineering Jalur 2 — Word2Vec (Skip-Gram, Deterministik, L2-Normalized)

Pelatihan model **Word2Vec** pada teks kalimat utuh (`df_w2v['cleaned_text']`) dengan spesifikasi:
- **Arsitektur**: Skip-gram (`sg=1`) dengan konteks `window=5`.
- **Dimensi & Epoch**: 100 dimensi vektor, dilatih selama 50 epoch (`epochs=50`).
- **Reproducibility 100%**: Menggunakan `workers=1`, `seed=42`, dan fungsi hash deterministik untuk eliminasi keacakan multithreading.
- **Normalisasi Vektor**: Rata-rata vektor kata dihitung per kalimat, kemudian diterapkan **L2 Unit Normalization** (`sklearn.preprocessing.normalize(..., norm='l2')`) per baris sehingga seluruh vektor dokumen memiliki norma Euclidean $= 1.0$.

In [4]:
# ── 1. Tokenisasi Kalimat Word2Vec ──────────────────────────────────────────
tokenized_w2v = [text.split() for text in df_w2v['cleaned_text'].fillna('')]

# ── 2. Fungsi Hash Deterministik untuk Gensim ────────────────────────────────
def deterministic_hash(astring: str) -> int:
    """Menjamin hash deterministik 32-bit di semua platform/OS."""
    return int.from_bytes(astring.encode('utf-8'), 'little') % (2**32)

# ── 3. Latih Model Word2Vec (Skip-gram, 100-dim, 50-epochs) ─────────────────
w2v_model = Word2Vec(
    sentences=tokenized_w2v,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    epochs=50,
    seed=SEED,
    workers=1,
    hashfxn=deterministic_hash
)

print(f"Model Word2Vec berhasil dilatih: {len(w2v_model.wv.key_to_index)} kosakata unik.")

# ── 4. Sentence Vector Aggregation & L2-Normalization ────────────────────────
def get_sentence_vector(words: list, model: Word2Vec) -> np.ndarray:
    valid_words = [w for w in words if w in model.wv.key_to_index]
    if valid_words:
        return np.mean(model.wv[valid_words], axis=0)
    return np.zeros(model.vector_size, dtype=np.float32)

raw_w2v_vectors = np.array([get_sentence_vector(words, w2v_model) for words in tokenized_w2v])

# L2 Unit Normalization per baris
w2v_normalized = normalize(raw_w2v_vectors, norm='l2')

df_w2v_features = pd.DataFrame(w2v_normalized, columns=[f"w2v_{i}" for i in range(100)])

# ── 5. Health Check: L2-Norm Validation ─────────────────────────────────────
norms = np.linalg.norm(w2v_normalized, axis=1)
print(f"[OK] Word2Vec selesai: {df_w2v_features.shape[1]} dimensi fitur.")
print(f"   * L2-Norm Min  : {norms.min():.4f}")
print(f"   * L2-Norm Max  : {norms.max():.4f}")
print(f"   * L2-Norm Mean : {norms.mean():.4f}")

assert np.allclose(norms, 1.0, atol=1e-5), "Peringatan: Vektor Word2Vec belum ternormalisasi L2 dengan benar!"
print(f"Bentuk matriks fitur Word2Vec: {df_w2v_features.shape}")

Model Word2Vec berhasil dilatih: 2230 kosakata unik.
[OK] Word2Vec selesai: 100 dimensi fitur.
   * L2-Norm Min  : 1.0000
   * L2-Norm Max  : 1.0000
   * L2-Norm Mean : 1.0000
Bentuk matriks fitur Word2Vec: (952, 100)


---
## 4. Pembuatan Dataset EDA/Arsip (Z-Score) vs. Dataset Training (Unscaled Anti-Leakage)

Untuk menjamin metodologi penelitian yang bebas kebocoran data (*leak-free*):
1. **Dataset Training PureText (104 Kolom)**:
   - Digunakan pada **Phase 1 (Text Representation Experiment)** untuk menguji murni kekuatan representasi teks (TF-IDF vs Word2Vec vs SBERT) tanpa fitur numerik domain agar model tidak menghafal formula prioritas.
   - Kolom: `['project_id', 'rank', 'relevance_grade', 'type_encoded', ...100 fitur teks...]`.
2. **Dataset Training Fusion (108 Kolom)**:
   - Digunakan pada **Feature Fusion & Ablation Study**. Fitur numerik domain `[value, effort, risk, stakeholder_priority]` dibiarkan **unscaled (skala asli 1–5)** agar proses scaling `StandardScaler` nantinya dilakukan *in-fold* di dalam Cross-Validation.
   - Kolom `priority_score` **wajib di-drop** untuk mencegah Target Leakage.
3. **Dataset EDA / Ready (113 Kolom)**:
   - Menyertakan seluruh metadata dan fitur numerik domain yang sudah di-transformasi dengan **`StandardScaler`** untuk visualisasi korelasi dan distribusi.

In [5]:
# ── 1. Metadata & Domain Features Definition ────────────────────────────────
domain_cols = ['value', 'effort', 'risk', 'stakeholder_priority']

# ── 2. Z-Score Normalization Khusus Dataset EDA / Arsip ──────────────────────
scaler = StandardScaler()
df_domain_scaled = pd.DataFrame(
    scaler.fit_transform(df_tfidf[domain_cols]),
    columns=domain_cols
)

# ── 3. Rakit Dataset EDA & Arsip Ready (Lengkap + Z-Score) ──────────────────
eda_base_cols = ['id', 'project_id', 'type', 'type_encoded', 'requirement_text', 'cleaned_text']
eda_target_cols = ['priority_score', 'rank', 'relevance_grade']

df_full_tfidf = pd.concat([
    df_tfidf[eda_base_cols],
    df_domain_scaled,
    df_tfidf[eda_target_cols],
    df_tfidf_features
], axis=1)

df_full_w2v = pd.concat([
    df_w2v[eda_base_cols],
    df_domain_scaled,
    df_w2v[eda_target_cols],
    df_w2v_features
], axis=1)

# ── 4. Rakit Dataset Training PureText (104 Kolom, Anti-Formula Leakage) ────
pure_base_cols = ['project_id', 'rank', 'relevance_grade', 'type_encoded']

df_train_tfidf_pure = pd.concat([df_tfidf[pure_base_cols], df_tfidf_features], axis=1)
df_train_w2v_pure   = pd.concat([df_w2v[pure_base_cols], df_w2v_features], axis=1)

# ── 5. Rakit Dataset Training Fusion (108 Kolom, Unscaled Numerical Features)
fusion_base_cols = ['project_id', 'rank', 'relevance_grade', 'type_encoded'] + domain_cols

df_train_tfidf_fusion = pd.concat([df_tfidf[fusion_base_cols], df_tfidf_features], axis=1)
df_train_w2v_fusion   = pd.concat([df_w2v[fusion_base_cols], df_w2v_features], axis=1)

# ── 6. Validation Gates: Schema & Leakage Check ─────────────────────────────
assert 'priority_score' not in df_train_tfidf_pure.columns, "Target leakage detected in PureText TFIDF!"
assert 'priority_score' not in df_train_tfidf_fusion.columns, "Target leakage detected in Fusion TFIDF!"
assert 'priority_score' not in df_train_w2v_pure.columns, "Target leakage detected in PureText W2V!"
assert 'priority_score' not in df_train_w2v_fusion.columns, "Target leakage detected in Fusion W2V!"

assert df_train_tfidf_pure.shape[1] == 104, f"Expected 104 cols in PureText, got {df_train_tfidf_pure.shape[1]}"
assert df_train_tfidf_fusion.shape[1] == 108, f"Expected 108 cols in Fusion, got {df_train_tfidf_fusion.shape[1]}"
assert df_full_tfidf.shape[1] == 113, f"Expected 113 cols in Full EDA, got {df_full_tfidf.shape[1]}"

print("[Gate 2 Passed] Seluruh skema dataset training bebas kebocoran target formula!")
print(f"  * Dimensi Training PureText : {df_train_tfidf_pure.shape} (104 kolom)")
print(f"  * Dimensi Training Fusion   : {df_train_tfidf_fusion.shape} (108 kolom)")
print(f"  * Dimensi Full EDA / Ready  : {df_full_tfidf.shape} (113 kolom)")

[Gate 2 Passed] Seluruh skema dataset training bebas kebocoran target formula!
  * Dimensi Training PureText : (952, 104) (104 kolom)
  * Dimensi Training Fusion   : (952, 108) (108 kolom)
  * Dimensi Full EDA / Ready  : (952, 113) (113 kolom)


---
## 5. Export Semua File ke Direktori Lokal (`./output_datasets/`)

Menyimpan seluruh 8 variasi file dataset ke disk lokal dalam folder `output_datasets/` tanpa dependensi Colab.

In [6]:
# ── 1. Export Seluruh Dataset ke Disk Lokal ─────────────────────────────────
exports = [
    # 1. Dataset Arsip Human-Readable (Excel)
    ("Dataset_Ready_TFIDF.xlsx", df_full_tfidf, "excel"),
    ("Dataset_Ready_Word2Vec.xlsx", df_full_w2v, "excel"),
    # 2. Dataset Lengkap untuk EDA (CSV)
    ("Dataset_EDA_TFIDF.csv", df_full_tfidf, "csv"),
    ("Dataset_EDA_Word2Vec.csv", df_full_w2v, "csv"),
    # 3. Dataset Training Phase 1: Murni Teks (CSV)
    ("Dataset_Training_TFIDF_PureText.csv", df_train_tfidf_pure, "csv"),
    ("Dataset_Training_Word2Vec_PureText.csv", df_train_w2v_pure, "csv"),
    # 4. Dataset Training Phase 2: Feature Fusion (CSV)
    ("Dataset_Training_TFIDF_Fusion.csv", df_train_tfidf_fusion, "csv"),
    ("Dataset_Training_Word2Vec_Fusion.csv", df_train_w2v_fusion, "csv"),
]

records = []
for filename, df_target, file_type in exports:
    target_path = OUTPUT_DIR / filename
    if file_type == "excel":
        df_target.to_excel(target_path, index=False)
    else:
        df_target.to_csv(target_path, index=False)
    
    size_kb = target_path.stat().st_size / 1024
    records.append({
        "Nama File": filename,
        "Dimensi (Baris, Kolom)": f"{df_target.shape[0]} x {df_target.shape[1]}",
        "Ukuran (KB)": f"{size_kb:.1f} KB",
        "Tipe Penggunaan": (
            "Arsip Excel" if "Ready" in filename else
            "Analisis EDA" if "EDA" in filename else
            "Training Murni Teks" if "PureText" in filename else
            "Training Feature Fusion"
        )
    })

summary_df = pd.DataFrame(records)
print("[OK] Ekspor Berhasil! Ringkasan File di './output_datasets/':")
print(summary_df.to_string(index=False))

[OK] Ekspor Berhasil! Ringkasan File di './output_datasets/':
                             Nama File Dimensi (Baris, Kolom) Ukuran (KB)         Tipe Penggunaan
              Dataset_Ready_TFIDF.xlsx              952 x 113    356.5 KB             Arsip Excel
           Dataset_Ready_Word2Vec.xlsx              952 x 113   1356.1 KB             Arsip Excel
                 Dataset_EDA_TFIDF.csv              952 x 113    662.9 KB            Analisis EDA
              Dataset_EDA_Word2Vec.csv              952 x 113   1433.2 KB            Analisis EDA
   Dataset_Training_TFIDF_PureText.csv              952 x 104    394.9 KB     Training Murni Teks
Dataset_Training_Word2Vec_PureText.csv              952 x 104   1122.1 KB     Training Murni Teks
     Dataset_Training_TFIDF_Fusion.csv              952 x 108    402.3 KB Training Feature Fusion
  Dataset_Training_Word2Vec_Fusion.csv              952 x 108   1129.6 KB Training Feature Fusion


---
## 6. Pipeline Helper: In-Fold Feature Extraction & Scaling (Phase 1 Training)

Kode helper ini mendemonstrasikan cara meletakkan `StandardScaler` secara **in-fold** di dalam loop Cross-Validation (misalnya dengan `GroupKFold` pada `project_id`) untuk mencegah kebocoran data (*pre-split data leakage*) saat melatih model Learning-to-Rank.

In [7]:
# ── Helper In-Fold Feature Scaler & Cross-Validation Template ────────────────
def prepare_fold_data(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols: list,
    scale_numeric: bool = True
):
    """
    Contoh helper fungsi untuk memproses fitur numerik secara in-fold:
    - Melakukan fitting StandardScaler hanya pada train_df.
    - Mentransformasi val_df menggunakan parameter dari train_df.
    """
    X_train = train_df[feature_cols].copy()
    X_val   = val_df[feature_cols].copy()
    
    if scale_numeric:
        num_cols = [c for c in ['value', 'effort', 'risk', 'stakeholder_priority'] if c in feature_cols]
        if num_cols:
            fold_scaler = StandardScaler()
            X_train[num_cols] = fold_scaler.fit_transform(X_train[num_cols])
            X_val[num_cols]   = fold_scaler.transform(X_val[num_cols])
            
    return X_train, X_val

# Demonstrasi pemisahan GroupKFold (5-Fold) berbasis project_id
gkf = GroupKFold(n_splits=5)
groups = df_train_tfidf_fusion['project_id']

print("Demonstrasi 5-Fold GroupKFold Split:")
for fold, (train_idx, val_idx) in enumerate(gkf.split(df_train_tfidf_fusion, groups=groups), start=1):
    train_proj = df_train_tfidf_fusion.iloc[train_idx]['project_id'].unique()
    val_proj   = df_train_tfidf_fusion.iloc[val_idx]['project_id'].unique()
    # Pastikan tidak ada overlap proyek
    assert len(set(train_proj).intersection(set(val_proj))) == 0
    print(f"  Fold {fold}: Train={len(train_idx)} baris ({len(train_proj)} proyek), Val={len(val_idx)} baris ({len(val_proj)} proyek)")

print("\nHelper in-fold scaling siap diintegrasikan langsung ke script eksperimen MLflow Phase 1!")

Demonstrasi 5-Fold GroupKFold Split:
  Fold 1: Train=721 baris (29 proyek), Val=231 baris (1 proyek)
  Fold 2: Train=775 baris (26 proyek), Val=177 baris (4 proyek)
  Fold 3: Train=769 baris (23 proyek), Val=183 baris (7 proyek)
  Fold 4: Train=774 baris (21 proyek), Val=178 baris (9 proyek)
  Fold 5: Train=769 baris (21 proyek), Val=183 baris (9 proyek)

Helper in-fold scaling siap diintegrasikan langsung ke script eksperimen MLflow Phase 1!
